![Piggy bank](piggy_bank.jpg)

Personal loans are a lucrative revenue stream for banks. The typical interest rate of a two-year loan in the United Kingdom is [around 10%](https://www.experian.com/blogs/ask-experian/whats-a-good-interest-rate-for-a-personal-loan/). This might not sound like a lot, but in September 2022 alone UK consumers borrowed [around £1.5 billion](https://www.ukfinance.org.uk/system/files/2022-12/Household%20Finance%20Review%202022%20Q3-%20Final.pdf), which would mean approximately £300 million in interest generated by banks over two years!

You have been asked to work with a bank to clean the data they collected as part of a recent marketing campaign, which aimed to get customers to take out a personal loan. They plan to conduct more marketing campaigns going forward so would like you to ensure it conforms to the specific structure and data types that they specify so that they can then use the cleaned data you provide to set up a PostgreSQL database, which will store this campaign's data and allow data from future campaigns to be easily imported. 

They have supplied you with a csv file called `"bank_marketing.csv"`, which you will need to clean, reformat, and split the data, saving three final csv files. Specifically, the three files should have the names and contents as outlined below:

## `client.csv`

| column | data type | description | cleaning requirements |
|--------|-----------|-------------|-----------------------|
| `client_id` | `integer` | Client ID | N/A |
| `age` | `integer` | Client's age in years | N/A |
| `job` | `object` | Client's type of job | Change `"."` to `"_"` |
| `marital` | `object` | Client's marital status | N/A |
| `education` | `object` | Client's level of education | Change `"."` to `"_"` and `"unknown"` to `np.NaN` |
| `credit_default` | `bool` | Whether the client's credit is in default | Convert to `boolean` data type:<br> `1` if `"yes"`, otherwise `0` |
| `mortgage` | `bool` | Whether the client has an existing mortgage (housing loan) | Convert to boolean data type:<br> `1` if `"yes"`, otherwise `0` |

<br>

## `campaign.csv`

| column | data type | description | cleaning requirements |
|--------|-----------|-------------|-----------------------|
| `client_id` | `integer` | Client ID | N/A |
| `number_contacts` | `integer` | Number of contact attempts to the client in the current campaign | N/A |
| `contact_duration` | `integer` | Last contact duration in seconds | N/A |
| `previous_campaign_contacts` | `integer` | Number of contact attempts to the client in the previous campaign | N/A |
| `previous_outcome` | `bool` | Outcome of the previous campaign | Convert to boolean data type:<br> `1` if `"success"`, otherwise `0`. |
| `campaign_outcome` | `bool` | Outcome of the current campaign | Convert to boolean data type:<br> `1` if `"yes"`, otherwise `0`. |
| `last_contact_date` | `datetime` | Last date the client was contacted | Create from a combination of `day`, `month`, and a newly created `year` column (which should have a value of `2022`); <br> **Format =** `"YYYY-MM-DD"` |

<br>

## `economics.csv`

| column | data type | description | cleaning requirements |
|--------|-----------|-------------|-----------------------|
| `client_id` | `integer` | Client ID | N/A |
| `cons_price_idx` | `float` | Consumer price index (monthly indicator) | N/A |
| `euribor_three_months` | `float` | Euro Interbank Offered Rate (euribor) three-month rate (daily indicator) | N/A |

In [11]:
import pandas as pd
import numpy as np

# Start coding here...
df = pd.read_csv("bank_marketing.csv")

# First, let's see what columns we have
print("Available columns in the dataset:")
print(df.columns.tolist())
print("\nFirst few rows of the dataset:")
print(df.head())

# 1. Create client DataFrame
client = pd.DataFrame()
client['client_id'] = df['client_id']
client['age'] = df['age']

# Clean job column: replace '.' with '_'
client['job'] = df['job'].str.replace('.', '_', regex=False)

client['marital'] = df['marital']

# Clean education column: replace '.' with '_' and 'unknown' with NaN
client['education'] = df['education'].str.replace('.', '_', regex=False)
client['education'] = client['education'].replace('unknown', np.NaN)

# Convert credit_default to boolean (True for 'yes', False otherwise)
client['credit_default'] = (df['credit_default'] == 'yes')

# Convert mortgage to boolean (True for 'yes', False otherwise)
client['mortgage'] = (df['mortgage'] == 'yes')

print("\nClient DataFrame data types:")
print(client.dtypes)

# 2. Create campaign DataFrame
campaign = pd.DataFrame()
campaign['client_id'] = df['client_id']

# Based on the output from the previous step, we need to use the correct column names
# Let's check if these columns exist in the dataframe
required_campaign_cols = ['campaign', 'duration', 'previous', 'previous_outcome', 'campaign_outcome', 'month', 'day']
available_cols = [col for col in required_campaign_cols if col in df.columns]
print(f"\nAvailable campaign-related columns: {available_cols}")
print(f"Missing columns: {[col for col in required_campaign_cols if col not in df.columns]}")

# If the column names are different, we need to map them correctly
# Common alternative column names in bank marketing datasets:
# - 'campaign' might be 'nr_contacts' or 'num_contacts'
# - 'duration' might be 'dur' or 'last_contact_duration'
# - 'previous' might be 'prev_contacts' or 'previous_contacts'
# - 'previous_outcome' might be 'poutcome'
# - 'campaign_outcome' might be 'y' or 'outcome'

# Let's find the correct column names
col_mapping = {
    'number_contacts': None,  # Will be mapped to actual column name
    'contact_duration': None,
    'previous_campaign_contacts': None,
    'previous_outcome': None,
    'campaign_outcome': None
}

# Try to find the column for number of contacts
for possible_name in ['campaign', 'nr_contacts', 'num_contacts', 'contacts']:
    if possible_name in df.columns:
        col_mapping['number_contacts'] = possible_name
        break

# Try to find the column for duration
for possible_name in ['duration', 'dur', 'last_contact_duration']:
    if possible_name in df.columns:
        col_mapping['contact_duration'] = possible_name
        break

# Try to find the column for previous contacts
for possible_name in ['previous', 'prev_contacts', 'previous_contacts', 'prev']:
    if possible_name in df.columns:
        col_mapping['previous_campaign_contacts'] = possible_name
        break

# For previous_outcome, we know it exists from the value counts output
if 'previous_outcome' in df.columns:
    col_mapping['previous_outcome'] = 'previous_outcome'
elif 'poutcome' in df.columns:
    col_mapping['previous_outcome'] = 'poutcome'

# For campaign_outcome, we know it exists from the value counts output
if 'campaign_outcome' in df.columns:
    col_mapping['campaign_outcome'] = 'campaign_outcome'
elif 'y' in df.columns:
    col_mapping['campaign_outcome'] = 'y'
elif 'outcome' in df.columns:
    col_mapping['campaign_outcome'] = 'outcome'

print(f"\nColumn mapping found: {col_mapping}")

# Now assign the columns based on the mapping
if col_mapping['number_contacts']:
    campaign['number_contacts'] = df[col_mapping['number_contacts']]
else:
    # If we can't find the column, we need to handle it
    print("WARNING: Could not find column for number_contacts")
    campaign['number_contacts'] = 0  # Default value

if col_mapping['contact_duration']:
    campaign['contact_duration'] = df[col_mapping['contact_duration']]
else:
    print("WARNING: Could not find column for contact_duration")
    campaign['contact_duration'] = 0  # Default value

if col_mapping['previous_campaign_contacts']:
    campaign['previous_campaign_contacts'] = df[col_mapping['previous_campaign_contacts']]
else:
    print("WARNING: Could not find column for previous_campaign_contacts")
    campaign['previous_campaign_contacts'] = 0  # Default value

# Convert previous_outcome to boolean (True for 'success', False otherwise)
if col_mapping['previous_outcome']:
    campaign['previous_outcome'] = (df[col_mapping['previous_outcome']] == 'success')
else:
    campaign['previous_outcome'] = False

# Convert campaign_outcome to boolean (True for 'yes', False otherwise)
if col_mapping['campaign_outcome']:
    campaign['campaign_outcome'] = (df[col_mapping['campaign_outcome']] == 'yes')
else:
    campaign['campaign_outcome'] = False

# Create last_contact_date from day, month, and year (2022)
# First ensure month is properly formatted (2 digits)
if 'month' in df.columns and 'day' in df.columns:
    months = df['month'].map({
        'jan': 1, 'feb': 2, 'mar': 3, 'apr': 4, 'may': 5, 'jun': 6,
        'jul': 7, 'aug': 8, 'sep': 9, 'oct': 10, 'nov': 11, 'dec': 12
    })
    
    # Create date strings and convert to datetime
    date_strings = '2022-' + months.astype(str).str.zfill(2) + '-' + df['day'].astype(str).str.zfill(2)
    campaign['last_contact_date'] = pd.to_datetime(date_strings, format='%Y-%m-%d')
else:
    print("WARNING: Could not find month and day columns")
    campaign['last_contact_date'] = pd.NaT

print("\nCampaign DataFrame data types:")
print(campaign.dtypes)

# 3. Create economics DataFrame
economics = pd.DataFrame()
economics['client_id'] = df['client_id']
economics['cons_price_idx'] = df['cons_price_idx']
economics['euribor_three_months'] = df['euribor_three_months']

# Save the three DataFrames to CSV files without index
client.to_csv('client.csv', index=False)
campaign.to_csv('campaign.csv', index=False)
economics.to_csv('economics.csv', index=False)

print("\nFiles saved successfully!")
print(f"client.csv shape: {client.shape}")
print(f"campaign.csv shape: {campaign.shape}")
print(f"economics.csv shape: {economics.shape}")

Available columns in the dataset:
['client_id', 'age', 'job', 'marital', 'education', 'credit_default', 'mortgage', 'month', 'day', 'contact_duration', 'number_contacts', 'previous_campaign_contacts', 'previous_outcome', 'cons_price_idx', 'euribor_three_months', 'campaign_outcome']

First few rows of the dataset:
   client_id  age  ... euribor_three_months campaign_outcome
0          0   56  ...                4.857               no
1          1   57  ...                4.857               no
2          2   37  ...                4.857               no
3          3   40  ...                4.857               no
4          4   56  ...                4.857               no

[5 rows x 16 columns]

Client DataFrame data types:
client_id          int64
age                int64
job               object
marital           object
education         object
credit_default      bool
mortgage            bool
dtype: object

Available campaign-related columns: ['previous_outcome', 'campaign_outcome',

In [12]:
df = pd.read_csv("bank_marketing.csv")

for col in ["credit_default", "mortgage", "previous_outcome", "campaign_outcome"]:
    print(col)
    print("--------------")
    print(df[col].value_counts())

credit_default
--------------
no         32588
unknown     8597
yes            3
Name: credit_default, dtype: int64
mortgage
--------------
yes        21576
no         18622
unknown      990
Name: mortgage, dtype: int64
previous_outcome
--------------
nonexistent    35563
failure         4252
success         1373
Name: previous_outcome, dtype: int64
campaign_outcome
--------------
no     36548
yes     4640
Name: campaign_outcome, dtype: int64
